# Reproducing *Optimizing YOLOv8 for Parking Space Detection*Trains and evaluates all five YOLOv8 variants from[arXiv:2505.17364](https://arxiv.org/abs/2505.17364) on the PKLot dataset, then diffs yournumbers against the paper's Table 3 (accuracy) and Table 4 (complexity).**Variants:** `yolov8n` (baseline) · `resnet18` · `vgg16` · `efficientnetv2` · `ghostp2`### How to use this1. **Runtime → Change runtime type → GPU** (T4 is fine, A100 matches the paper).2. Run cells top to bottom.3. First time through, set `QUICK_TEST = True` in the config cell. That runs 1 epoch on 5% of   the data (~5 min) and proves the whole pipeline works before you commit hours to it.4. Then set `QUICK_TEST = False` and run the real thing.### TimingFull 5-model run is **~3.5 h on an A100**, and roughly **2–3× that on a free T4**. Colab willdisconnect before that finishes. Two mitigations are built in:* results are written straight to Google Drive, so nothing is lost on a disconnect;* already-finished models are skipped, and interrupted ones resume from `last.pt`.So you can just re-run the notebook after a disconnect, or set `MODELS_TO_RUN` to one variantper session.### Known deviations from the authors' repoThe committed configs in `pokhrelapar/yolov8-pklot` are missing the `scales:` block, which makesthem build models ~4× larger than the paper's Table 4 describes. The configs written by thisnotebook restore it and have been verified to reproduce Table 4. The VGG-16 config was nevercommitted at all and is reconstructed here. Details in `PAPER_NOTES.md` §7.

## 0 · Environment check

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "!! No GPU. Runtime -> Change runtime type -> GPU, then re-run.")
import torch
print(f"torch {torch.__version__} | cuda {torch.version.cuda} | available={torch.cuda.is_available()}")

In [ ]:
# The paper used Ultralytics 8.3.x with PyTorch 2.6. Pinning keeps the TorchVision/Index module
# semantics stable; set to "" to take whatever is latest.
ULTRALYTICS_VERSION = "8.3.100"

spec = f"ultralytics=={ULTRALYTICS_VERSION}" if ULTRALYTICS_VERSION else "ultralytics"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], capture_output=True, text=True)
if r.returncode:
    print(f"Pin {spec} failed, falling back to latest.\n{r.stderr[-800:]}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import ultralytics
ultralytics.checks()

## 1 · ConfigurationThe only cell you normally need to edit.

In [ ]:
from pathlib import Path

# ----- which variants to run -------------------------------------------------------------
# Trim this list to train one model per Colab session if you keep getting disconnected.
MODELS_TO_RUN = ["yolov8n", "resnet18", "vgg16", "efficientnetv2", "ghostp2"]

# ----- pipeline smoke test ---------------------------------------------------------------
QUICK_TEST = True          # True -> 1 epoch on 5% of the data. Set False for the real run.

# ----- training hyperparameters (paper Table 2 + the authors' logged args.yaml) -----------
EPOCHS     = 20
BATCH      = 16
IMGSZ      = 640
SEED       = 0             # authors logged seed=0, deterministic=true
WORKERS    = 8             # args.yaml says 8; the paper's Table 2 says 2 (paper is wrong)
OPTIMIZER  = "auto"        # -> AdamW(lr=0.001667, momentum=0.9), which is what the paper reports

# ----- where results go ------------------------------------------------------------------
USE_DRIVE     = True       # strongly recommended: survives runtime disconnects
DRIVE_PROJECT = "/content/drive/MyDrive/pklot-repro"
LOCAL_PROJECT = "/content/pklot-repro"

# ----- resume behaviour ------------------------------------------------------------------
FORCE_RETRAIN = False      # True -> retrain even if best.pt already exists
RESUME        = True       # True -> continue an interrupted run from last.pt

# ----- dataset ---------------------------------------------------------------------------
# "roboflow"  : download PKLot v2 (640) directly. Needs a free API key from roboflow.com.
# "drive_zip" : unzip PKLot.v2-640.yolov8.zip from your Drive (what the authors did).
# "existing"  : already extracted somewhere in this runtime.
DATASET_MODE     = "roboflow"
ROBOFLOW_API_KEY = ""      # <-- paste your key here for DATASET_MODE="roboflow"
DRIVE_ZIP        = "/content/drive/MyDrive/PKLot.v2-640.yolov8.zip"
EXISTING_DIR     = "/content/PKLot"

# The authors renamed Roboflow's space-empty/space-occupied to e/o (see their confusion
# matrices). Cosmetic only - class indices are unchanged.
RENAME_CLASSES_TO_EO = True

# -----------------------------------------------------------------------------------------
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
else:
    PROJECT = LOCAL_PROJECT
Path(PROJECT).mkdir(parents=True, exist_ok=True)

FRACTION   = 0.05 if QUICK_TEST else 1.0
RUN_EPOCHS = 1    if QUICK_TEST else EPOCHS

print(f"results -> {PROJECT}")
print(f"models  -> {MODELS_TO_RUN}")
print(f"mode    -> {'QUICK TEST (1 epoch, 5% data)' if QUICK_TEST else f'FULL RUN ({EPOCHS} epochs)'}")

## 2 · DatasetPKLot, Roboflow export **v2 ("640")** — 12,416 images, 2 classes, 70/20/10 split, alreadyresized to 640×640. This is the exact export the authors used (`PKLot.v2-640.yolov8.zip`).The cell also rewrites `data.yaml` to absolute paths. Roboflow ships relative ones(`../train/images`) which silently break under Colab.

In [ ]:
import os, zipfile, yaml, glob, shutil

DATA_ROOT = Path("/content/PKLot")

if DATASET_MODE == "roboflow":
    assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY in the config cell (free at roboflow.com)."
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
    from roboflow import Roboflow
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    (rf.workspace("brad-dwyer").project("pklot-1tros")
       .version(2).download("yolov8", location=str(DATA_ROOT)))

elif DATASET_MODE == "drive_zip":
    assert Path(DRIVE_ZIP).exists(), f"Not found: {DRIVE_ZIP}"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP) as z:
        z.extractall(DATA_ROOT)

elif DATASET_MODE == "existing":
    DATA_ROOT = Path(EXISTING_DIR)
    assert DATA_ROOT.exists(), f"Not found: {DATA_ROOT}"

else:
    raise ValueError(DATASET_MODE)

# --- locate and normalise data.yaml -------------------------------------------------------
candidates = sorted(DATA_ROOT.rglob("data.yaml"))
assert candidates, f"No data.yaml found under {DATA_ROOT}"
DATA_YAML = candidates[0]
base = DATA_YAML.parent
d = yaml.safe_load(DATA_YAML.read_text())

for split, folders in [("train", ["train"]), ("val", ["valid", "val"]), ("test", ["test"])]:
    found = next((base / f / "images" for f in folders if (base / f / "images").is_dir()), None)
    assert found, f"Missing image folder for split '{split}' under {base}"
    d[split] = str(found)
d.pop("path", None)

names = d.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
assert len(names) == 2, f"Expected 2 classes, got {names}"
if RENAME_CLASSES_TO_EO:
    names = ["e", "o"]
d["names"] = names
d["nc"] = 2

DATA_YAML.write_text(yaml.safe_dump(d, sort_keys=False))
DATA_YAML = str(DATA_YAML)

print(yaml.safe_dump(d, sort_keys=False))
for split in ("train", "val", "test"):
    n_img = len(glob.glob(os.path.join(d[split], "*")))
    n_lbl = len(glob.glob(os.path.join(d[split].replace("/images", "/labels"), "*.txt")))
    print(f"{split:5s}  {n_img:6,} images  {n_lbl:6,} labels")
print("\nPaper expects roughly: train 8,691 / val 2,483 / test 1,242  (70/20/10 of 12,416)")

## 3 · Model configurationsWritten out here so the notebook is self-contained. All four carry the `scales:` block that theauthors' committed YAMLs are missing — without it Ultralytics builds the PANet neck at`depth=width=1.0` instead of `0.33/0.25`, giving a model ~4× larger than Table 4 reports.`yolov8n` needs no file; it uses the stock `yolov8n.pt`.

In [ ]:
CFG_DIR = Path("/content/cfg"); CFG_DIR.mkdir(exist_ok=True)

_TV_HEAD = """
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]          # cat backbone P4
  - [-1, 3, C2f, [512]]                # 7
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 1], 1, Concat, [1]]          # cat backbone P3
  - [-1, 3, C2f, [256]]                # 10 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 7], 1, Concat, [1]]          # cat head P4
  - [-1, 3, C2f, [512]]                # 13 (P4/16-medium)
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 4], 1, Concat, [1]]          # cat head P5
  - [-1, 3, C2f, [1024]]               # 16 (P5/32-large)
  - [[10, 13, 16], 1, Detect, [nc]]    # Detect(P3, P4, P5)
"""

def _torchvision_cfg(tv_name, c2_out, taps):
    """taps = [(channels, split_index), ...] for P3/8, P4/16, P5/32."""
    lines = "\n".join(f"  - [0, 1, Index, [{c}, {i}]]" for c, i in taps)
    return (
        "nc: 2\n"
        "scales:\n"
        "  n: [0.33, 0.25, 1024]\n"
        "backbone:\n"
        f'  - [-1, 1, TorchVision, [{c2_out}, "{tv_name}", "DEFAULT", True, 2, True]]\n'
        f"{lines}\n"
        "  - [-1, 1, SPPF, [1024, 5]]\n"
        + _TV_HEAD
    )

# Split indices verified against torchvision: y[0] is the input, y[i] the output of sub-module i-1.
CFGS = {
    # ResNet-18: layer2/layer3/layer4                                    (paper 5.4)
    "yolov8n-resnet18.yaml":
        _torchvision_cfg("resnet18", 2048, [(128, 6), (256, 7), (512, 8)]),
    # EfficientNetV2-S: features[3]/features[5]/features[7]              (paper 5.5)
    "yolov8n-efficientnetv2_S.yaml":
        _torchvision_cfg("efficientnet_v2_s", 2048, [(64, 4), (160, 6), (1280, 8)]),
    # VGG-16: features[22]/features[29]/features[30]  -- RECONSTRUCTED   (paper 5.6)
    "yolov8n-vgg16.yaml":
        _torchvision_cfg("vgg16", 512, [(512, 23), (512, 30), (512, 31)]),
}

# Ghost-P2 is Ultralytics' stock config: GhostConv/C3Ghost everywhere plus a 4th head at P2/4.
CFGS["yolov8n-ghost-p2.yaml"] = """nc: 2
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]            # 0-P1/2
  - [-1, 1, GhostConv, [128, 3, 2]]      # 1-P2/4
  - [-1, 3, C3Ghost, [128, True]]
  - [-1, 1, GhostConv, [256, 3, 2]]      # 3-P3/8
  - [-1, 6, C3Ghost, [256, True]]
  - [-1, 1, GhostConv, [512, 3, 2]]      # 5-P4/16
  - [-1, 6, C3Ghost, [512, True]]
  - [-1, 1, GhostConv, [1024, 3, 2]]     # 7-P5/32
  - [-1, 3, C3Ghost, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]             # 9
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [512]]              # 12
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [256]]              # 15 (P3/8-small)
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [128]]              # 18 (P2/4-xsmall)
  - [-1, 1, GhostConv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [256]]              # 21 (P3/8-small)
  - [-1, 1, GhostConv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [512]]              # 24 (P4/16-medium)
  - [-1, 1, GhostConv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [1024]]             # 27 (P5/32-large)
  - [[18, 21, 24, 27], 1, Detect, [nc]]  # Detect(P2, P3, P4, P5)
"""

for fname, text in CFGS.items():
    (CFG_DIR / fname).write_text(text)
    print("wrote", CFG_DIR / fname)

# key -> (display name, what to initialise training from)
VARIANTS = {
    "yolov8n":        ("YOLOv8n",             "yolov8n.pt"),
    "resnet18":       ("YOLO-ResNet-18",      str(CFG_DIR / "yolov8n-resnet18.yaml")),
    "vgg16":          ("YOLO-VGG16",          str(CFG_DIR / "yolov8n-vgg16.yaml")),
    "efficientnetv2": ("YOLO-EfficientNetV2", str(CFG_DIR / "yolov8n-efficientnetv2_S.yaml")),
    "ghostp2":        ("YOLO-Ghost-P2",       str(CFG_DIR / "yolov8n-ghost-p2.yaml")),
}
# Architecture spec for the complexity table. Same as above except yolov8n, which trains from
# .pt weights but has to be *built* from yaml to count params/layers/GFLOPs.
ARCH_CFG = {k: v[1] for k, v in VARIANTS.items()}
ARCH_CFG["yolov8n"] = "yolov8n.yaml"

## 4 · Architecture check — reproduces Table 4 without trainingBuilds each model at `nc=2`, runs one forward pass to prove the shapes line up, and reportsparams / layers / GFLOPs against the paper. **Run this before training** — a shape mismatchhere costs seconds, the same mismatch after an hour of training costs an hour.Downloads ImageNet weights for the three torchvision backbones (~650 MB total, VGG-16 is mostof it). They're cached and reused during training, so this is not wasted.

In [ ]:
import gc, io, contextlib, pandas as pd
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops

# Paper Table 4
PAPER_T4 = {
    "yolov8n":        dict(params_M=3.01, layers=129, gflops=8.2,   infer_ms=0.9),
    "resnet18":       dict(params_M=13.32, layers=132, gflops=35.2, infer_ms=9.0),
    "vgg16":          dict(params_M=17.78, layers=113, gflops=262.1, infer_ms=3.3),
    "efficientnetv2": dict(params_M=23.40, layers=564, gflops=56.4, infer_ms=4.1),
    "ghostp2":        dict(params_M=1.60, layers=290, gflops=8.8,   infer_ms=1.5),
}

rows = []
for key in MODELS_TO_RUN:
    with contextlib.redirect_stdout(io.StringIO()):
        m = DetectionModel(ARCH_CFG[key], nc=2, verbose=False).eval()
    with torch.no_grad():
        out = m(torch.zeros(1, 3, IMGSZ, IMGSZ))
    feats = out[1] if isinstance(out, (list, tuple)) else out
    n_l = sum(1 for _, mm in m.named_modules() if len(mm._modules) == 0)
    n_p = sum(p.numel() for p in m.parameters())
    fl = get_flops(m, IMGSZ)
    p = PAPER_T4[key]
    rows.append({
        "model": VARIANTS[key][0],
        "params_M": round(n_p / 1e6, 2), "paper_params_M": p["params_M"],
        "layers": n_l,                   "paper_layers": p["layers"],
        "GFLOPs": round(fl, 1),          "paper_GFLOPs": p["gflops"],
        "head_strides": [IMGSZ // t.shape[-1] for t in feats],
    })
    del m; gc.collect()

arch_df = pd.DataFrame(rows)
arch_df["params_delta_%"] = ((arch_df.params_M / arch_df.paper_params_M - 1) * 100).round(1)
arch_df["GFLOPs_delta_%"] = ((arch_df.GFLOPs / arch_df.paper_GFLOPs - 1) * 100).round(1)
display(arch_df)
print("\nhead_strides shows the detection scales: [8,16,32] normally, [4,8,16,32] for Ghost-P2 (extra P2 head).")

## 5 · TrainEach variant trains under identical settings, exactly as the paper specifies. Everything landsin `PROJECT/<key>_train/`.Safe to re-run: finished models are skipped, interrupted ones resume from `last.pt`.

In [ ]:
import time, traceback
from ultralytics import YOLO

train_log = {}
for key in MODELS_TO_RUN:
    label, init = VARIANTS[key]
    run_dir = Path(PROJECT) / f"{key}_train"
    best, last = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"

    if best.exists() and not FORCE_RETRAIN:
        print(f"[skip] {label}: {best} already exists")
        train_log[key] = "skipped"
        continue

    print(f"\n{'=' * 78}\n  TRAIN  {label}   ({init})\n{'=' * 78}")
    t0 = time.time()
    try:
        if last.exists() and RESUME:
            print(f"resuming from {last}")
            YOLO(str(last)).train(resume=True)
        else:
            YOLO(init).train(
                data=DATA_YAML, epochs=RUN_EPOCHS, batch=BATCH, imgsz=IMGSZ,
                seed=SEED, deterministic=True, workers=WORKERS, optimizer=OPTIMIZER,
                fraction=FRACTION, project=PROJECT, name=f"{key}_train",
                exist_ok=True, plots=True, val=True,
            )
        train_log[key] = f"ok ({time.time() - t0:.0f}s)"
    except Exception:
        traceback.print_exc()
        train_log[key] = "FAILED"
    print(f"[{key}] {train_log[key]}")

print("\n" + "\n".join(f"{k:16s} {v}" for k, v in train_log.items()))

## 6 · Evaluate on the test split`results.csv` only ever logs **validation**. The paper's Table 3 is the **test** split, producedby a separate `model.val(split="test")`. This cell collects both so you can compare like withlike, and re-times every model on this one GPU (the paper's ms column mixes A100 and T4 runs).

In [ ]:
import pandas as pd

results = []
for key in MODELS_TO_RUN:
    label = VARIANTS[key][0]
    best = Path(PROJECT) / f"{key}_train" / "weights" / "best.pt"
    if not best.exists():
        print(f"[skip] {label}: no best.pt"); continue

    print(f"\n--- val(split='test')  {label} ---")
    model = YOLO(str(best))
    r = model.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=BATCH,
                  project=PROJECT, name=f"{key}_test", exist_ok=True, plots=True)

    row = {"key": key, "model": label,
           "precision": r.box.mp, "recall": r.box.mr,
           "mAP50": r.box.map50, "mAP50_95": r.box.map,
           "infer_ms": r.speed["inference"]}

    csv = Path(PROJECT) / f"{key}_train" / "results.csv"
    if csv.exists():
        hist = pd.read_csv(csv)
        hist.columns = [c.strip() for c in hist.columns]   # some versions pad with spaces
        last = hist.iloc[-1]
        for out_col, src in [("val_precision", "metrics/precision(B)"),
                             ("val_recall",    "metrics/recall(B)"),
                             ("val_mAP50",     "metrics/mAP50(B)"),
                             ("val_mAP50_95",  "metrics/mAP50-95(B)"),
                             ("train_time_s",  "time")]:
            row[out_col] = last[src] if src in last.index else float("nan")
    results.append(row)

res_df = pd.DataFrame(results).round(5)
res_df.to_csv(Path(PROJECT) / "my_results.csv", index=False)
display(res_df)

## 7 · Compare against the paperThree columns per metric:* **yours** — this run, test split* **paper** — Table 3* **repo log** — the authors' own `results.csv`, final epoch, **val** splitThe repo log is the more honest target: it is raw logged output rather than a retyped table, andit disagrees with Table 3 in two places (VGG-16 recall, and which model wins mAP50:95).Expect ±0.002 on precision/recall/mAP50 — they're saturated near 0.99 and that's just run noise.**mAP50:95 is the only metric here with real dynamic range** (0.896 → 0.986), so judge thereproduction on that.

In [ ]:
# Paper Table 3 (test split)
PAPER_T3 = {
    "yolov8n":        dict(precision=0.996, recall=0.996, mAP50=0.994, mAP50_95=0.970),
    "resnet18":       dict(precision=0.998, recall=0.997, mAP50=0.994, mAP50_95=0.976),
    "vgg16":          dict(precision=0.998, recall=0.985, mAP50=0.991, mAP50_95=0.985),
    "efficientnetv2": dict(precision=0.998, recall=0.997, mAP50=0.994, mAP50_95=0.986),
    "ghostp2":        dict(precision=0.968, recall=0.978, mAP50=0.991, mAP50_95=0.896),
}
# Authors' repo, final epoch of results.csv (val split)
REPO_VAL = {
    "yolov8n":        dict(precision=0.99773, recall=0.99797, mAP50=0.99448, mAP50_95=0.96595),
    "resnet18":       dict(precision=0.99846, recall=0.99838, mAP50=0.99455, mAP50_95=0.97269),
    "vgg16":          dict(precision=0.99845, recall=0.99869, mAP50=0.99454, mAP50_95=0.98613),
    "efficientnetv2": dict(precision=0.99874, recall=0.99865, mAP50=0.99460, mAP50_95=0.98269),
    "ghostp2":        dict(precision=0.96797, recall=0.98073, mAP50=0.99207, mAP50_95=0.89303),
}

rows = []
for _, r in res_df.iterrows():
    k = r["key"]
    for metric in ("precision", "recall", "mAP50", "mAP50_95"):
        rows.append({
            "model": r["model"], "metric": metric,
            "yours_test": round(r[metric], 4),
            "paper_T3": PAPER_T3[k][metric],
            "delta_vs_paper": round(r[metric] - PAPER_T3[k][metric], 4),
            "yours_val": round(r.get(f"val_{metric}", float("nan")), 4),
            "repo_val": round(REPO_VAL[k][metric], 4),
            "delta_vs_repo": round(r.get(f"val_{metric}", float("nan")) - REPO_VAL[k][metric], 4),
        })
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(Path(PROJECT) / "comparison_vs_paper.csv", index=False)

for metric in ("mAP50_95", "mAP50", "precision", "recall"):
    print(f"\n===== {metric} =====")
    display(cmp_df[cmp_df.metric == metric].drop(columns="metric").reset_index(drop=True))

print("\n===== ranking by mAP50:95 =====")
mine = res_df[["model", "mAP50_95"]].sort_values("mAP50_95", ascending=False)
print("yours (test): " + "  >  ".join(f"{m} {v:.4f}" for m, v in mine.values))
print("paper Tbl 3 : YOLO-EfficientNetV2 0.986 > YOLO-VGG16 0.985 > YOLO-ResNet-18 0.976 "
      "> YOLOv8n 0.970 > YOLO-Ghost-P2 0.896")
print("repo logs   : YOLO-VGG16 0.9861 > YOLO-EfficientNetV2 0.9827 > YOLO-ResNet-18 0.9727 "
      "> YOLOv8n 0.9660 > YOLO-Ghost-P2 0.8930")
print("\nNote the paper and the authors' own logs disagree on the top two. Your run breaks the tie.")

print("\n===== inference latency, all timed on THIS GPU =====")
lat = res_df[["model", "infer_ms"]].copy()
lat["paper_ms"] = [PAPER_T4[k]["infer_ms"] for k in res_df["key"]]
display(lat)
print("The paper's ms column mixes A100 and T4 runs (it lists ResNet-18 at 9.0 ms/35 GFLOPs vs "
      "VGG16 at 3.3 ms/262 GFLOPs, which is impossible on one device). Trust your column.")

## 8 · Training curvesReproduces the paper's Figures 10–14 plus precision/recall.

In [ ]:
import matplotlib.pyplot as plt

METRICS = [
    ("train/box_loss", "Train box loss"),
    ("train/cls_loss", "Train classification loss"),
    ("val/box_loss", "Validation box loss"),
    ("val/cls_loss", "Validation classification loss"),
    ("metrics/mAP50(B)", "mAP@50"),
    ("metrics/mAP50-95(B)", "mAP@50:95"),
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)", "Recall"),
]

curves = {}
for key in MODELS_TO_RUN:
    csv = Path(PROJECT) / f"{key}_train" / "results.csv"
    if csv.exists():
        df = pd.read_csv(csv)
        df.columns = [c.strip() for c in df.columns]
        curves[VARIANTS[key][0]] = df

if not curves:
    print("No results.csv yet - train something first.")
else:
    fig, axes = plt.subplots(4, 2, figsize=(13, 18))
    for ax, (col, title) in zip(axes.ravel(), METRICS):
        for label, df in curves.items():
            if col in df:
                ax.plot(df["epoch"], df[col], marker="o", ms=3, label=label)
        ax.set_title(title); ax.set_xlabel("epoch"); ax.grid(alpha=.3)
        # Ghost-P2's val/cls_loss diverges to ~65 in the authors' logs and flattens everything else
        if col == "val/cls_loss":
            vals = [v for df in curves.values() if col in df for v in df[col]]
            if vals and max(vals) / max(min(vals), 1e-6) > 50:
                ax.set_yscale("log"); ax.set_title(title + "  (log scale)")
        ax.legend(fontsize=8)
    plt.tight_layout()
    out = Path(PROJECT) / "comparison_plots.png"
    plt.savefig(out, dpi=130, bbox_inches="tight")
    print("saved", out)
    plt.show()

## 9 · ExportEverything is already on Drive if `USE_DRIVE=True`. This just bundles it for download.

In [ ]:
import shutil
archive = shutil.make_archive("/content/pklot_repro_results", "zip", PROJECT)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("Download it from the file browser instead:", e)